# OpenRouter provider

Goal: probe `finstack_ai.providers.openrouter.is_available()`, construct `Agent.openrouter` for OpenRouter's stateless Responses endpoint, show `:nitro`/`:floor` model-routing suffixes, note the `media_tools` toolset flag, then optionally run a live model and tool call with an explicit `api_key`.

Trust: T1 native provider plus T2 Python ports. Network: skipped unless the next cell or `OPENROUTER_API_KEY` has a key.

`api_key` is required and keyword-only. The binding does not read environment variables. Paste a key below, or keep using the environment.

Set `OPENROUTER_MODEL` to an OpenRouter model id. `reasoning_effort` is optional (`none`, `minimal`, `low`, `medium`, `high`, `xhigh`, `max`), and `reasoning_summary` is optional (`auto`, `concise`, `detailed`). `referer` and `title` set non-secret `HTTP-Referer`/`X-Title` attribution headers. Requests target `https://openrouter.ai/api/v1/responses` with `store: false`.

In [ ]:
OPENROUTER_API_KEY = ""  # paste a valid key to run the live cells; never commit it
OPENROUTER_MODEL = "openai/gpt-5"
OPENROUTER_REFERER = ""  # optional non-secret HTTP-Referer
OPENROUTER_TITLE = ""  # optional non-secret X-Title
OPENROUTER_REASONING_EFFORT = "low"  # or none / minimal / medium / high / xhigh / max
OPENROUTER_REASONING_SUMMARY = "auto"  # or concise / detailed

## Availability probe

`finstack_ai.providers` is a lazy subpackage; each linked provider exposes an `is_available()` probe backed by `linked_providers()`. This wheel links `openai`, `anthropic`, `ollama`, and `openrouter`, so both checks below should agree.

In [ ]:
import finstack_ai
import finstack_ai.providers as providers

print("openrouter is_available:", providers.openrouter.is_available())
print("linked_providers:", finstack_ai.linked_providers())

## Construction and capability catalog

The live cell uses `OPENROUTER_MODEL`, `OPENROUTER_REFERER`, `OPENROUTER_TITLE`, `OPENROUTER_REASONING_EFFORT`, and `OPENROUTER_REASONING_SUMMARY` from the first code cell.

In [ ]:
from _support import live_value

api_key = live_value(OPENROUTER_API_KEY, "OPENROUTER_API_KEY")
if api_key:
    agent = await finstack_ai.Agent.openrouter(
        OPENROUTER_MODEL,
        "Answer concisely.",
        api_key=api_key,
        referer=OPENROUTER_REFERER or None,
        title=OPENROUTER_TITLE or None,
        reasoning_effort=OPENROUTER_REASONING_EFFORT or None,
        reasoning_summary=OPENROUTER_REASONING_SUMMARY or None,
    )
    print(agent.compact_capability_catalog())
else:
    print("skipped: OPENROUTER_API_KEY unset")

## Provider routing

OpenRouter's routing controls (`provider`, e.g. `{"order": ["openai"], "sort": "throughput"}`, or a `models` fallback array) pass through Rust `ModelSettings` untouched, but the Python binding does not expose a per-run settings argument — `Agent.run` takes `input`, `timeout_seconds`, `max_cycles`, `max_output_retries`, `capability`, and `attachments` only, with no `settings`/`model_settings` parameter. From Python, request routing through the `:nitro` (highest throughput) or `:floor` (lowest price) model-name suffixes instead, appended directly to the model id passed to `Agent.openrouter`.

In [ ]:
if api_key:
    nitro_agent = await finstack_ai.Agent.openrouter(
        f"{OPENROUTER_MODEL}:nitro",
        "Answer concisely.",
        api_key=api_key,
        reasoning_effort=OPENROUTER_REASONING_EFFORT or None,
        reasoning_summary=OPENROUTER_REASONING_SUMMARY or None,
    )
    print(nitro_agent.compact_capability_catalog())
else:
    print("skipped: OPENROUTER_API_KEY unset")

## Media generation toolset

`media_tools=True` registers the OpenRouter media-generation toolset (image, speech, video, and transcription tools) alongside the model, reusing `api_key`, `referer`, and `title`. This cell only constructs the agent to show the toolset in the capability catalog; it does not call a tool or spend media credits.

In [ ]:
if api_key:
    media_agent = await finstack_ai.Agent.openrouter(
        OPENROUTER_MODEL,
        "Answer concisely.",
        api_key=api_key,
        media_tools=True,
    )
    print(media_agent.compact_capability_catalog())
else:
    print("skipped: OPENROUTER_API_KEY unset")

## Tool call

A rejected or revoked key causes the run to fail before model output begins.

In [ ]:
from pydantic import BaseModel


class Answer(BaseModel):
    answer: int


@finstack_ai.tool
def add(left: int, right: int) -> Answer:
    """Add two integers."""
    return Answer(answer=left + right)


tools = finstack_ai.pydantic_toolset(
    add,
    component="notebook.toolset.openrouter",
    name="math",
)

if api_key:
    reasoning_result = await agent.run(
        "What is 20 plus 22? Reply with only the number."
    )
    print("reasoning:", reasoning_result.text)

    tool_agent = await finstack_ai.Agent.openrouter(
        OPENROUTER_MODEL,
        "Answer with the tool result only.",
        api_key=api_key,
        reasoning_effort=OPENROUTER_REASONING_EFFORT or None,
        reasoning_summary=OPENROUTER_REASONING_SUMMARY or None,
        toolsets=[tools],
    )
    tool_result = await tool_agent.run("Add 20 and 22")
    print("tool:", tool_result.text)
else:
    print("skipped: OPENROUTER_API_KEY unset")